In [1]:
import sys
sys.path.append("..")

from src.preprocess import preprocess

text = "APOE4 expression in iPSC-derived neurons results in increased tau phosphorylation."
print(preprocess(text))

['apoe4', 'express', 'ipsc', 'deriv', 'neuron', 'result', 'increas', 'tau', 'phosphoryl']


In [2]:
print("No stemming:       ", preprocess(text, stem=False))
print("No stopword removal:", preprocess(text, remove_stopwords=False))
print("Full:              ", preprocess(text))

No stemming:        ['apoe4', 'expression', 'ipsc', 'derived', 'neurons', 'results', 'increased', 'tau', 'phosphorylation']
No stopword removal: ['apoe4', 'express', 'in', 'ipsc', 'deriv', 'neuron', 'result', 'in', 'increas', 'tau', 'phosphoryl']
Full:               ['apoe4', 'express', 'ipsc', 'deriv', 'neuron', 'result', 'increas', 'tau', 'phosphoryl']


In [3]:
from beir.datasets.data_loader import GenericDataLoader
corpus, queries, qrels = GenericDataLoader("../data/scifact").load(split="test")

qid = list(queries)[0]
print(queries[qid])
print(preprocess(queries[qid]))


d:\search_keyword_semantic\venv\Lib\site-packages\beir\datasets\data_loader.py:8: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm
100%|██████████| 5183/5183 [00:00<00:00, 117641.43it/s]

0-dimensional biomaterials show inductive properties.
['0', 'dimension', 'biomateri', 'show', 'induct', 'properti']


In [4]:
import sys
sys.path.append("..")

import math
from collections import Counter
from beir.datasets.data_loader import GenericDataLoader
from src.preprocess import preprocess

corpus, queries, qrels = GenericDataLoader("../data/scifact").load(split="test")
docs = {did: (d["title"] + ". " + d["text"]).strip() for did, d in corpus.items()}

doc_ids = list(docs)
tokenized = [preprocess(docs[d]) for d in doc_ids]
print(tokenized[0][:15])

100%|██████████| 5183/5183 [00:00<00:00, 46473.96it/s]


['microstructur', 'develop', 'human', 'newborn', 'cerebr', 'white', 'matter', 'assess', 'vivo', 'diffus', 'tensor', 'magnet', 'reson', 'imag', 'alter']


In [5]:
N = len(tokenized)

# document frequency: in how many documents does each word appear?
df = Counter()
for toks in tokenized:
    df.update(set(toks))

# inverse document frequency (smoothed)
idf = {t: math.log((1 + N) / (1 + c)) + 1 for t, c in df.items()}

# TF-IDF vector for each document, stored as a dict {word: weight}
doc_vecs = []
for toks in tokenized:
    tf = Counter(toks)
    vec = {t: c * idf[t] for t, c in tf.items()}
    norm = math.sqrt(sum(w * w for w in vec.values()))
    doc_vecs.append({t: w / norm for t, w in vec.items()})

print("Vocabulary size:", len(idf))

Vocabulary size: 26067


In [6]:
def tfidf_search(query, k=10):
    tf = Counter(preprocess(query))
    qvec = {t: c * idf[t] for t, c in tf.items() if t in idf}
    norm = math.sqrt(sum(w * w for w in qvec.values())) or 1.0
    qvec = {t: w / norm for t, w in qvec.items()}

    scores = []
    for i, dvec in enumerate(doc_vecs):
        s = sum(w * dvec.get(t, 0.0) for t, w in qvec.items())
        scores.append((doc_ids[i], s))
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:k]


In [9]:
for qid in list(queries)[:3]:
    print("QUERY:", queries[qid])
    relevant = set(qrels[qid])
    for rank, (did, score) in enumerate(tfidf_search(queries[qid], k=5), 1):
        mark = "✔" if did in relevant else " "
        print(f"  {rank}. {mark} {score:.3f} | {docs[did][:90]}")
    print()

QUERY: 0-dimensional biomaterials show inductive properties.
  1.   0.141 | Systematic review of role of bisphosphonates on skeletal morbidity in metastatic cancer.. 
  2.   0.122 | High-performance neuroprosthetic control by an individual with tetraplegia.. BACKGROUND Pa
  3.   0.115 | Body size and outcomes on peritoneal dialysis in the United States.. BACKGROUND Being over
  4.   0.108 | Fruit consumption and risk of type 2 diabetes: results from three prospective longitudinal
  5.   0.101 | The impact of hydroxyethyl starches in cardiac surgery: a meta-analysis. INTRODUCTION Rece

QUERY: 1,000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants.
  1.   0.423 | Common and rare variants in multifactorial susceptibility to common diseases. Here, we giv
  2.   0.353 | Rare and common variants: twenty arguments. Genome-wide association studies have greatly i
  3. ✔ 0.287 | Rare Variants Create Synt

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(analyzer=preprocess)    # reuse our preprocessing
X = vectorizer.fit_transform([docs[d] for d in doc_ids])

def sklearn_search(query, k=10):
    q = vectorizer.transform([query])
    sims = (X @ q.T).toarray().ravel()
    top = sims.argsort()[::-1][:k]
    return [(doc_ids[i], float(sims[i])) for i in top]

qid = list(queries)[0]
mine = [d for d, _ in tfidf_search(queries[qid])]
theirs = [d for d, _ in sklearn_search(queries[qid])]
print("Same top-10:", mine == theirs)


Same top-10: True


In [11]:
for qid, q in queries.items():
    top = {d for d, _ in sklearn_search(q, k=10)}
    if not (top & set(qrels[qid])):
        print("QUERY:", q)
        for did in qrels[qid]:
            print("RELEVANT DOC:", docs[did][:300])
        break
    

QUERY: 0-dimensional biomaterials show inductive properties.
RELEVANT DOC: New opportunities: the use of nanotechnologies to manipulate and track stem cells.. Nanotechnologies are emerging platforms that could be useful in measuring, understanding, and manipulating stem cells. Examples include magnetic nanoparticles and quantum dots for stem cell labeling and in vivo track
